# SheetalAI — Phase 1 data check
Renders the LST layer and key drivers from the aligned feature stack, and confirms hot zones land where built-up / low-vegetation areas are.

Run the pipeline first:
```bash
DATA_SOURCE=synthetic uv run python gee_export.py && uv run python features.py
```

In [ ]:
import os
import rasterio
import numpy as np
import matplotlib.pyplot as plt

CITY = os.environ.get('CITY', 'ahmedabad')
stack_path = f'../data/{CITY}/stack.tif'
ds = rasterio.open(stack_path)
bands = list(ds.descriptions)
print('bands:', bands)
data = ds.read(masked=True)
idx = {b: i for i, b in enumerate(bands)}

In [ ]:
lst = data[idx['lst_c']]
plt.figure(figsize=(9, 8))
im = plt.imshow(lst, cmap='inferno')
plt.colorbar(im, label='LST (°C)', shrink=0.8)
plt.title(f'{CITY.title()} — Land Surface Temperature')
plt.axis('off')
plt.tight_layout()
plt.show()
print('LST °C  min %.1f  mean %.1f  max %.1f' % (lst.min(), lst.mean(), lst.max()))

In [ ]:
# Drivers panel
show = ['ndvi', 'impervious_frac', 'albedo', 'dist_to_water', 'pop_density', 'vulnerability']
fig, axes = plt.subplots(2, 3, figsize=(14, 9))
for ax, b in zip(axes.ravel(), show):
    im = ax.imshow(data[idx[b]], cmap='viridis')
    ax.set_title(b)
    ax.axis('off')
    fig.colorbar(im, ax=ax, shrink=0.7)
plt.tight_layout()
plt.show()

In [ ]:
# Sanity: hottest decile should be more built-up / less vegetated than coolest decile
lst_flat = lst.compressed()
imp = data[idx['impervious_frac']].compressed()
ndvi = data[idx['ndvi']].compressed()
hot = lst_flat > np.percentile(lst_flat, 90)
cool = lst_flat < np.percentile(lst_flat, 10)
print('hottest 10%%: impervious=%.2f ndvi=%.2f' % (imp[hot].mean(), ndvi[hot].mean()))
print('coolest 10%%: impervious=%.2f ndvi=%.2f' % (imp[cool].mean(), ndvi[cool].mean()))